In [58]:
from core import utils, database, paths
import pandas as pd
import datetime
import requests
from plyer import notification

In [59]:
today = datetime.date.today().strftime('%Y-%m-%d')


In [60]:
try:
    data : pd.DataFrame | None = database.DatabaseManager().run_query("""select * from tesco_analysts.pmajor1_wtd_for_daily_tableau_test""")
    if data is None or (hasattr(data, "empty") and data.empty):
        raise Exception("nem sikerült")
    
except Exception as e:
    import traceback
    print(f'Hiba történt: {traceback.format_exc()}')

In [77]:
data.to_csv(fr"{paths.ASSETS_PATH}\AI_test\daily_test_data{today}.csv", index=False)

In [78]:
data['day_dt'] = pd.to_datetime(data['day'].astype(str), format='%Y%m%d')
max_day = data['day_dt'].max()

In [79]:
change_log = "Totally new calculation & prompt" 

In [ ]:
payload_options={
            "format": "json",
            "temperature": 0.12,
            "top_p": 0.95,
            "num_predict": 1000,    
            "num_ctx": 4096,
            "seed": 42,
            "repeat_penalty": 1.18,
            "repeat_last_n": 256,
            "presence_penalty": 0.1,
            "frequency_penalty": 0.1,
            "stop": ['}\n\n']     
        }

In [81]:
import os
import json

def get_and_increment_version(state_file: str, key: str = "version_number", start: int = 1) -> int:
    os.makedirs(os.path.dirname(state_file), exist_ok=True)

    state = {}
    if os.path.exists(state_file):
        try:
            with open(state_file, "r", encoding="utf-8") as f:
                state = json.load(f)
        except json.JSONDecodeError:
            state = {}

    current = int(state.get(key, start - 1))
    new_value = current + 1
    state[key] = new_value

    with open(state_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    return new_value

In [82]:
version_state_path = os.path.join(fr"{paths.ASSETS_PATH}\AI_test\ai_daily_summary_state_new.json")
version_number = get_and_increment_version(version_state_path, start=1)

In [83]:
def analyse_financial_summaries(system_prompt, user_data, model='martain7r/finance-llama-8b:q4_k_m',options=payload_options):
    url = "http://localhost:11434/api/chat" # Fontos: a /api/chat végpontot használjuk!
    
    
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_data}
        ],
        "stream": False,
        "options": options
}
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        
        return response.json().get("message", {}).get("content", "Nothing to analyze")
    else:
        return f"Error: {response.status_code} - {response.text}"

In [84]:
import pandas as pd

# -----------------------------------------------------------
# 1. LFL pár (LFL és LFL_LY) egy bármilyen dataframe szeletre
# -----------------------------------------------------------
def compute_lfl_pair(df: pd.DataFrame):
    def safe_sum(col):
        return df[col].sum() if col in df.columns else 0

    lfl = (
        safe_sum("sales_lfl_standard") +
        safe_sum("sales_lfl_promo") +
        safe_sum("sales_lfl_clearance") +
        safe_sum("sales_lfl_others")
    )

    lfl_ly = (
        safe_sum("sales_lflb_standard") +
        safe_sum("sales_lflb_promo") +
        safe_sum("sales_lflb_clearance") +
        safe_sum("sales_lflb_others")
    )

    return lfl, lfl_ly


# -----------------------------------------------------------
# 2. Headline LFL számítása
# -----------------------------------------------------------
def compute_headline_lfl(df: pd.DataFrame):
    total_lfl, total_lfl_ly = compute_lfl_pair(df)
    if total_lfl_ly == 0:
        return 0.0, total_lfl, total_lfl_ly
    pct = (total_lfl / total_lfl_ly - 1) * 100
    return pct, total_lfl, total_lfl_ly


# -----------------------------------------------------------
# 3. Országonként LFL
# -----------------------------------------------------------
def compute_country_lfl(df: pd.DataFrame):
    out = {}
    for c in ["CZ", "SK", "HU"]:
        c_df = df[df["country"] == c]
        if c_df.empty:
            continue
        lfl, lfl_ly = compute_lfl_pair(c_df)
        pct = 0.0 if lfl_ly == 0 else (lfl / lfl_ly - 1) * 100
        out[c] = pct
    return out


# -----------------------------------------------------------
# 4. Department szintű LFL + delta
# -----------------------------------------------------------
def compute_dept_lfl(df: pd.DataFrame):
    out = {}
    for dept in df["department_name"].unique():
        d_df = df[df["department_name"] == dept]
        lfl, lfl_ly = compute_lfl_pair(d_df)
        pct = 0 if lfl_ly == 0 else (lfl / lfl_ly - 1) * 100
        out[dept] = {
            "LFL": lfl,
            "LFL_LY": lfl_ly,
            "delta": lfl - lfl_ly,
            "LFL_pct": pct
        }
    return out


# -----------------------------------------------------------
# 5. Top/bottom 3 hozzájárulás
# -----------------------------------------------------------
def compute_top_bottom_contributions(dept_dict, total_lfl_ly):
    ordered = sorted(dept_dict.items(), key=lambda x: x[1]["delta"], reverse=True)
    top3 = ordered[:3]
    bottom3 = ordered[-3:]

    if total_lfl_ly == 0:
        return top3, bottom3, 0.0, 0.0

    top_contrib = sum(d[1]["delta"] for d in top3) / total_lfl_ly * 100
    bottom_contrib = sum(d[1]["delta"] for d in bottom3) / total_lfl_ly * 100

    return top3, bottom3, top_contrib, bottom_contrib

In [85]:
def build_report_input(df: pd.DataFrame, week: str, currency="£", region="3CE"):
    # --- Headline LFL ---
    headline_lfl_pct, total_lfl, total_lfl_ly = compute_headline_lfl(df)

    # --- Country LFL ---
    country_lfl_dict = compute_country_lfl(df)

    # --- Dept LFL + rankings ---
    dept_lfl_dict = compute_dept_lfl(df)
    top3, bottom3, top_contrib, bottom_contrib = compute_top_bottom_contributions(
        dept_lfl_dict, total_lfl_ly
    )

    # --- Total sales ---
    total_sales = (
        df["sales_ex_vat_ty_standard"].sum() +
        df["sales_ex_vat_ty_promo"].sum() +
        df["sales_ex_vat_ty_clearance"].sum() +
        (df["sales_ex_vat_ty_others"].sum() if "sales_ex_vat_ty_others" in df.columns else 0)
    )

    # --- Total margin ---
    total_margin = (
        df["margin_ty_standard"].sum() +
        df["margin_ty_promo"].sum() +
        df["margin_ty_clearance"].sum() +
        (df["margin_ty_others"].sum() if "margin_ty_others" in df.columns else 0)
    )
    margin_rate = 0 if total_sales == 0 else total_margin / total_sales * 100

    # --- TOTAL BUDGET & FORECAST ---
    total_budget = df["sales_budget"].drop_duplicates().sum()
    total_forecast = df["daily_sales_forecast"].drop_duplicates().sum()

    vs_budget_abs = total_sales - total_budget
    vs_budget_pct = 0 if total_budget == 0 else (total_sales / total_budget - 1) * 100

    vs_forecast_abs = total_sales - total_forecast
    vs_forecast_pct = 0 if total_forecast == 0 else (total_sales / total_forecast - 1) * 100

    # --- DEPARTMENT szintű budget/forecast variancia ---
    # --- DEPARTMENT szintű budget/forecast variancia ---
    dept_variances = {}

    for dept, stats in dept_lfl_dict.items():

        # Dept subset
        d_df = df[df["department_name"] == dept]

        # --- Dept sales (TY) ---
        dept_sales = (
            d_df["sales_ex_vat_ty_standard"].sum() +
            d_df["sales_ex_vat_ty_promo"].sum() +
            d_df["sales_ex_vat_ty_clearance"].sum() +
            (d_df["sales_ex_vat_ty_others"].sum() if "sales_ex_vat_ty_others" in d_df.columns else 0)
        )

        # --- Dept budget: csak uniq érték / nap (vagy akár egész file-ból)
        dept_budget = d_df["sales_budget"].drop_duplicates().sum()

        # --- Dept forecast: ugyanez uniq értékekből
        dept_forecast = d_df["daily_sales_forecast"].drop_duplicates().sum()

        # --- Varianciák ---
        dept_variances[dept] = {
            "sales": dept_sales,

            "budget": dept_budget,
            "vs_budget_abs": dept_sales - dept_budget,
            "vs_budget_pct": 0 if dept_budget == 0 else (dept_sales / dept_budget - 1) * 100,

            "forecast": dept_forecast,
            "vs_forecast_abs": dept_sales - dept_forecast,
            "vs_forecast_pct": 0 if dept_forecast == 0 else (dept_sales / dept_forecast - 1) * 100,
        }

    # --- Renderer input ---
    data = {
        "meta": {"week": week, "currency": currency, "region": region},

        "sales_week": {
            "total_sales": total_sales,
            "lfl_pct": headline_lfl_pct,
            "lfl_by_country": country_lfl_dict,
            "top_depts": [x[0] for x in top3],
            "top_depts_lfl_contrib_pct": top_contrib,
            "bottom_depts": [x[0] for x in bottom3],
            "bottom_depts_lfl_contrib_pct": bottom_contrib,

            "vs_budget_abs": vs_budget_abs,
            "vs_budget_pct": vs_budget_pct,
        },
        "sales_vs_bgt": {
            "vs_budget_abs" :vs_budget_abs,
            "vs_budget_pct" : vs_budget_pct
        },

        "sales_vs_fct": {
            "sales_var_abs": vs_forecast_abs,
            "sales_var_pct": vs_forecast_pct,
        },

        "dept_variance": dept_variances,

        "scan_margin": {
            "scan_margin_rate_pct": margin_rate,
            "scan_margin_value": total_margin,
            "vs_fct_abs": vs_forecast_abs,
            "vs_fct_pct": vs_forecast_pct,
        },

        "price": {}
    }

    return data

In [86]:
# ============================================================
#                    RENDERER MODUL (B option)
#     + Top/Bottom 3 departments vs Budget and Forecast
# ============================================================

def fmt_money(v: float, currency: str = "£") -> str:
    if abs(v) >= 1e6:
        return f"{currency}{v/1e6:.1f}M"
    elif abs(v) >= 1e3:
        return f"{currency}{v/1e3:.0f}k"
    return f"{currency}{v:.0f}"

def fmt_pct(v: float) -> str:
    return f"{v:.1f}%"

def join_list(items: list) -> str:
    if len(items) == 0:
        return ""
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]

# ---------------- SALES WEEK ----------------
def render_sales_week(d: dict, currency: str):
    out = []
    out.append(
        f"Sales for yesterday was {fmt_money(d['total_sales'], currency)} "
        f"with {fmt_pct(d['lfl_pct'])} LFL overall.\n"
    )

    # COUNTRY LFL
    if "lfl_by_country" in d:
        c = d["lfl_by_country"]
        parts = []
        for k in ["CZ", "SK", "HU"]:
            if k in c:
                parts.append(f"{k}: {fmt_pct(c[k])}")
        if parts:
            out.append("Country LFL :\n " + ", ".join(parts) + ".")

    # TOP / BOTTOM DEPARTMENTS (LFL)
    if d.get("top_depts"):
        out.append(
            f"Top departments driving sales LFL are {join_list(d['top_depts'])}, "
            f"contributing {fmt_pct(d['top_depts_lfl_contrib_pct'])}."
        )

    if d.get("bottom_depts"):
        out.append(
            f"Departments with the weakest LFL performance are {join_list(d['bottom_depts'])}, "
            f"dragging total LFL by {fmt_pct(d['bottom_depts_lfl_contrib_pct'])}."
        )

    # TOTAL VS BUDGET
    if "vs_budget_abs" in d:
        diff = d["vs_budget_abs"]
        status = "ahead of budget" if diff >= 0 else "behind budget"
        out.append(
            f"Total sales were {status} by {fmt_money(abs(diff), currency)} "
            f"({fmt_pct(d['vs_budget_pct'])})."
        )

    return " ".join(out)



# ---------------- SALES vs BUDGET ----------------
def render_sales_vs_budget(d: dict, region: str, currency: str):
    diff = d["vs_budget_abs"]
    status = "ahead of budget" if diff >= 0 else "behind budget"
    return (
        f"{region} sales were {status} by {fmt_money(abs(diff), currency)} "
        f"({fmt_pct(d['vs_budget_pct'])})."
    )


# ---------------- SALES vs FORECAST ----------------
def render_sales_vs_fct(d: dict, region: str, currency: str):
    diff = d["sales_var_abs"]
    status = "ahead of forecast" if diff >= 0 else "behind forecast"
    return (
        f"{region} sales were {status} by {fmt_money(abs(diff), currency)} "
        f"({fmt_pct(d['sales_var_pct'])})."
    )


# ---------------- SCAN MARGIN ----------------
def render_scan_margin(d: dict, region: str, currency: str):
    out = []
    out.append(f"{region} Scan margin rate was {fmt_pct(d['scan_margin_rate_pct'])}.")
    diff = d["vs_fct_abs"]
    status = "higher" if diff >= 0 else "lower"
    out.append(
        f"Scan margin was {status} than forecast by {fmt_money(abs(diff), currency)} "
        f"({fmt_pct(d['vs_fct_pct'])})."
    )
    return " ".join(out)


# ---------------- NEW: DEPARTMENT VARIANCE SECTION ----------------
def render_dept_variance_section(dept_var: dict, currency: str):
    """Top/Bottom 3 departments vs budget & forecast."""

    if not dept_var:
        return ""

    # ---- VS BUDGET ----
    sorted_budget = sorted(
        dept_var.items(),
        key=lambda x: x[1]["vs_budget_abs"],
        reverse=True
    )
    top3_budget = sorted_budget[:3]
    bottom3_budget = sorted_budget[-3:]

    # ---- VS FORECAST ----
    sorted_fct = sorted(
        dept_var.items(),
        key=lambda x: x[1]["vs_forecast_abs"],
        reverse=True
    )
    top3_fct = sorted_fct[:3]
    bottom3_fct = sorted_fct[-3:]

    out = ["Top and bottom performing departments vs Budget and Forecast:"]

    # --- BUDGET TOP ---
    out.append(
        "• Ahead of budget: " +
        ", ".join([
            f"{d[0]} ({fmt_pct(d[1]['vs_budget_pct'])})"
            for d in top3_budget
        ]) + "."
    )

    # --- BUDGET BOTTOM ---
    out.append(
        "• Behind budget: " +
        ", ".join([
            f"{d[0]} ({fmt_pct(d[1]['vs_budget_pct'])})"
            for d in bottom3_budget
        ]) + "."
    )

    # --- FORECAST TOP ---
    out.append(
        "• Ahead of forecast: " +
        ", ".join([
            f"{d[0]} ({fmt_pct(d[1]['vs_forecast_pct'])})"
            for d in top3_fct
        ]) + "."
    )

    # --- FORECAST BOTTOM ---
    out.append(
        "• Behind forecast: " +
        ", ".join([
            f"{d[0]} ({fmt_pct(d[1]['vs_forecast_pct'])})"
            for d in bottom3_fct
        ]) + "."
    )

    return " ".join(out)


# ---------------- MASTER RENDER ----------------
def render_report(data: dict):
    out = []

    meta = data["meta"]
    region = meta["region"]
    week = meta["week"]
    currency = meta["currency"]

    out.append(f"{region} sales report for week {week}")

    # SALES WEEK
    out.append("\nSALES PERFORMANCE\n" +
               render_sales_week(data["sales_week"], currency))

    # BUDGET VARIANCE
    
    out.append("\nBUDGET VARIANCE\n" +
            render_sales_vs_budget(data["sales_vs_bgt"], region, currency))


    # FORECAST VARIANCE
    out.append("\nFORECAST VARIANCE\n" +
               render_sales_vs_fct(data["sales_vs_fct"], region, currency))

    # SCAN MARGIN
    out.append("\nSCAN MARGIN\n" +
               render_scan_margin(data["scan_margin"], region, currency))

    # DEPARTMENT VARIANCE
    out.append("\nDEPARTMENT PERFORMANCE VS BUDGET AND FORECAST\n" +
               render_dept_variance_section(data["dept_variance"], currency))

    return "\n".join(out)

In [88]:
# --- CSV betöltés ---
df = pd.read_csv(fr"{paths.ASSETS_PATH}\AI_test\daily_test_data{today}.csv")

# Csak az utolsó napra szűrés (ahogy kérted)
df['day_dt'] = pd.to_datetime(df['day'].astype(str), format='%Y%m%d')
last_day = df['day_dt'].max()
df_last = df[df['day_dt'] == last_day]

# --- Adatstruktúra előállítása ---
report_data = build_report_input(df_last, week=max_day)

# --- Riport szöveg generálása ---
result = render_report(report_data)

In [89]:
system_message = ("""
 You are a STRICT POST-EDITING ENGINE. 
Your ONLY task is to rewrite the text so it reads more naturally and fluently 
while keeping EVERYTHING else EXACTLY the same.

""")

In [90]:
user_message = f"""

You are a STRICT NO-CHANGE BUSINESS REPORT POST-EDITOR.

Your job is ONLY to make each sentence sound more natural and fluent, 
WITHOUT changing ANY facts, numbers, order, structure, or meaning.

ABSOLUTE, UNBREAKABLE RULES:
- Keep the EXACT same number of sentences.
- Keep ALL sentences in the EXACT same order.
- Do NOT combine sentences.
- Do NOT split sentences.
- Do NOT remove sentences.
- Do NOT add sentences.
- Do NOT reorder anything.
- Do NOT add any interpretation, commentary, conclusions, adjectives, or opinions.
- Do NOT change ANY numbers, currencies, percentages, values, or department/country names.
- Do NOT change directional meaning (ahead/behind, up/down, stronger/weaker).
- Do NOT summarize or expand. 
- Do NOT invent content.

ALLOWED:
- Minimal grammar and flow adjustments INSIDE each sentence ONLY.
- Sentences must remain SEMANTICALLY IDENTICAL.

PROCESS:
1. Rewrite EACH sentence INDIVIDUALLY.
2. Each output sentence must correspond EXACTLY to one input sentence.
3. The output must contain the SAME number of paragraphs as the input.
4. The output must contain the SAME number of sentences per paragraph.
5. Do NOT modify punctuation in ways that change readability of numbers (e.g. “3.6%” must remain “3.6%”).

FORMAT:
Return ONLY the rewritten text with the EXACT SAME structural layout.

TEXT TO REWRITE:
 {result}"""

In [93]:
answer = analyse_financial_summaries(system_prompt=system_message, user_data=user_message, model='martain7r/finance-llama-8b:q4_k_m')

In [94]:
print(answer)

 Sales for yesterday was £697k with -2.0% LFL overall.
 Country LFL :
 CZ: -11.8%, SK: -12.8%, HU: 8.4%.
 Top departments driving sales LFL are Electrical, Home seasonal and Shopping bag, contributing 2.3%.
 Departments with the weakest LFL performance are WIGIG, Gardening and Car, dragging total LFL by -3.2%.
 Total sales were behind budget by £23k (-3.2%).


BUDGET VARIANCE
Sales were behind budget by £23k (-3.2%).

FORECAST VARIANCE
Sales were behind forecast by £32k (-4.4%).

SCAN MARGIN
Scan margin rate was 27.0%. Scan margin was lower than forecast by £32k (-4.4%).

DEPARTMENT PERFORMANCE VS BUDGET AND FORECAST
Top and bottom performing departments vs Budget and Forecast:
• Ahead of budget: Toys & Nursery (6.7%), Electrical (17.2%), Home seasonal (245.4%).
• Behind budget: Stationery (-9.1%), Home Entertainment (-22.9%), Gardening (-21.4%).
• Ahead of forecast: Electrical (32.9%), Home (7.5%), Shopping bag (17.0%).
• Behind forecast: Home Entertainment (-13.3%), Toys & Nursery (-

In [95]:
print(result)

3CE sales report for week 2026-02-09 00:00:00

SALES PERFORMANCE
Sales for yesterday was £697k with -2.0% LFL overall.
 Country LFL :
 CZ: -11.8%, SK: -12.8%, HU: 8.4%. Top departments driving sales LFL are Electrical, Home seasonal and Shopping bag, contributing 2.3%. Departments with the weakest LFL performance are WIGIG, Gardening and Car, dragging total LFL by -3.2%. Total sales were behind budget by £23k (-3.2%).

BUDGET VARIANCE
3CE sales were behind budget by £23k (-3.2%).

FORECAST VARIANCE
3CE sales were behind forecast by £32k (-4.4%).

SCAN MARGIN
3CE Scan margin rate was 27.0%. Scan margin was lower than forecast by £32k (-4.4%).

DEPARTMENT PERFORMANCE VS BUDGET AND FORECAST
Top and bottom performing departments vs Budget and Forecast: • Ahead of budget: Toys & Nursery (6.7%), Electrical (17.2%), Home seasonal (245.4%). • Behind budget: Stationery (-9.1%), Home Entertainment (-22.9%), Gardening (-21.4%). • Ahead of forecast: Electrical (32.9%), Home (7.5%), Shopping bag (1

In [96]:
log_file=f"Payload options:\n {payload_options}\nsystem prompt:\n{system_message}\nPrompt:\n{user_message}"

In [97]:
def write_out(content):
    output_file = os.path.join(fr"{paths.ASSETS_PATH}\AI_test\ai_daily_summary_v5{str(version_number)}.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Daily CE Performance Report - {max_day.date()}\n\n")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről

write_out(answer)

In [98]:
def write_out(content,version_number=version_number):
    output_file = os.path.join(fr"{paths.ASSETS_PATH}\AI_test\ai_daily_summary_5{str(version_number)}_options.md")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(f"# Version of trial - {str(version_number)}\n\n{change_log}\n\n{payload_options}\n\n{system_message}\n\n{user_message}")
        f.write(content.strip()) # strip() eltávolítja a felesleges üres sorokat a végéről
    print(f"Report saved: {output_file}")